# 🧠 Face Detection & Recognition System
### MTCNN + ArcFace + Cosine Similarity Pipeline
**Stack:** MTCNN · InsightFace (ArcFace) · OpenCV · Scikit-Learn · Pickle · Streamlit

---


## Step 1 — Install Required Packages

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "opencv-python",
    "mtcnn",
    "tensorflow",
    "numpy",
    "scikit-learn",
    "insightface",
    "onnxruntime",
    "pillow",
    "tqdm",
    "streamlit",
    "scipy",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages installed successfully!")


## Step 2 — Import Libraries

In [ ]:
import os
import cv2
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image

# Face Detection
from mtcnn import MTCNN

# Face Recognition
from insightface.app import FaceAnalysis

# ML utilities
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")


## Step 3 — Create Project Directory Structure

In [ ]:
# Create required folders
dirs = ["dataset", "embeddings", "models", "test_images", "output"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("📁 Project structure created:")
for d in dirs:
    print(f"   ✅ {d}/")

print("""
Face_Detection_Recognition/
├── dataset/          ← Put person folders here (e.g. dataset/John/)
├── embeddings/       ← Saved face embeddings (.pkl)
├── models/           ← Model cache
├── test_images/      ← Images to test recognition
└── output/           ← Annotated output images
""")


## Step 4 — Dataset Collection Helper

Place your face images inside `dataset/<PersonName>/`.  
For best accuracy follow these guidelines:

| Recommendation | Detail |
|---|---|
| Images per person | **20 – 50** |
| Angles | Front, left, right, slight up/down |
| Lighting | Indoor, outdoor, different brightness |
| Expressions | Neutral, smile, serious |
| Resolution | ≥ 100 × 100 px |

The cell below auto-captures from your webcam if no images exist yet.


In [ ]:
def capture_faces_from_webcam(person_name: str, num_images: int = 30):
    """Capture face images from webcam for a given person."""
    save_dir = Path("dataset") / person_name
    save_dir.mkdir(parents=True, exist_ok=True)

    detector = MTCNN()
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Webcam not accessible in this environment. Add images manually.")
        return

    count = 0
    print(f"📸 Capturing {num_images} images for '{person_name}' — press Q to quit early.")

    while count < num_images:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(rgb)

        for face in results:
            x, y, w, h = face["box"]
            x, y = max(0, x), max(0, y)
            conf = face["confidence"]

            if conf < 0.95:          # only high-confidence detections
                continue

            # Add 20 % padding for better ArcFace embedding quality
            pad_x = int(w * 0.2)
            pad_y = int(h * 0.2)
            x1 = max(0, x - pad_x)
            y1 = max(0, y - pad_y)
            x2 = min(frame.shape[1], x + w + pad_x)
            y2 = min(frame.shape[0], y + h + pad_y)

            face_img = frame[y1:y2, x1:x2]
            img_path = save_dir / f"{count:04d}.jpg"
            cv2.imwrite(str(img_path), face_img)
            count += 1

            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, f"{count}/{num_images}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        cv2.imshow("Capturing — press Q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"✅ Saved {count} images to {save_dir}")


# ── Uncomment and change the name to capture your own face ──
# capture_faces_from_webcam("YourName", num_images=30)

# ── Check existing dataset ──
dataset_path = Path("dataset")
persons = [p.name for p in dataset_path.iterdir() if p.is_dir()] if dataset_path.exists() else []
print(f"👥 Persons in dataset: {persons if persons else 'None — add images to dataset/<Name>/'}")
for p in persons:
    imgs = list((dataset_path / p).glob("*.jpg")) + list((dataset_path / p).glob("*.png"))
    print(f"   {p}: {len(imgs)} images")


## Step 5 — Initialize Models (MTCNN + ArcFace)

In [ ]:
# ── MTCNN (detection) ──────────────────────────────────────
mtcnn_detector = MTCNN()
print("✅ MTCNN loaded")

# ── InsightFace / ArcFace (recognition) ────────────────────
face_analyzer = FaceAnalysis(
    name="buffalo_l",           # buffalo_l = highest accuracy model
    root="models",
    providers=["CPUExecutionProvider"]   # change to CUDAExecutionProvider if GPU available
)
face_analyzer.prepare(ctx_id=0, det_size=(640, 640))
print("✅ ArcFace (buffalo_l) loaded")


## Step 6 — Helper Functions

In [ ]:
def preprocess_image(img_bgr: np.ndarray) -> np.ndarray:
    """Resize + histogram-equalise for better embedding quality."""
    img = cv2.resize(img_bgr, (640, 640))
    # CLAHE on luminance channel for lighting robustness
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def get_arcface_embedding(img_bgr: np.ndarray) -> np.ndarray | None:
    """Return L2-normalised 512-d ArcFace embedding or None if no face found."""
    img_pre = preprocess_image(img_bgr)
    faces = face_analyzer.get(img_pre)
    if not faces:
        return None
    # Pick the largest face
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    emb = face.embedding
    return normalize([emb])[0]          # L2 normalise


def detect_faces_mtcnn(img_bgr: np.ndarray, min_conf: float = 0.95):
    """Detect faces with MTCNN; return list of (x,y,w,h,conf) tuples."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = mtcnn_detector.detect_faces(rgb)
    boxes = []
    for r in results:
        if r["confidence"] >= min_conf:
            boxes.append((*r["box"], r["confidence"]))
    return boxes


def cosine_score(emb1: np.ndarray, emb2: np.ndarray) -> float:
    return float(cosine_similarity([emb1], [emb2])[0][0])


print("✅ Helper functions defined")


## Step 7 — Build Face Embeddings Database

In [ ]:
def build_embeddings_database(dataset_dir: str = "dataset",
                               save_path: str = "embeddings/face_embeddings.pkl",
                               augment: bool = True):
    """
    For every person folder, extract ArcFace embeddings from all images.
    Averages multiple embeddings per person → more robust identity vector.
    Supports basic augmentation (horizontal flip) to double data.
    """
    dataset_path = Path(dataset_dir)
    database: dict[str, np.ndarray] = {}

    persons = [p for p in dataset_path.iterdir() if p.is_dir()]
    if not persons:
        print("⚠️  No person folders found in dataset/. Add images first.")
        return {}

    for person_dir in persons:
        name = person_dir.name
        image_files = list(person_dir.glob("*.jpg")) + \
                      list(person_dir.glob("*.jpeg")) + \
                      list(person_dir.glob("*.png"))

        if not image_files:
            print(f"⚠️  No images found for {name}, skipping.")
            continue

        embeddings_list = []
        print(f"\n🔍 Processing: {name} ({len(image_files)} images)")

        for img_path in tqdm(image_files, desc=name):
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            emb = get_arcface_embedding(img)
            if emb is not None:
                embeddings_list.append(emb)

            # Augmentation: horizontal flip
            if augment:
                flipped = cv2.flip(img, 1)
                emb_f = get_arcface_embedding(flipped)
                if emb_f is not None:
                    embeddings_list.append(emb_f)

        if embeddings_list:
            # Mean embedding → robust identity representation
            mean_emb = normalize([np.mean(embeddings_list, axis=0)])[0]
            database[name] = mean_emb
            print(f"   ✅ {name}: {len(embeddings_list)} embeddings → 1 mean vector")
        else:
            print(f"   ❌ No valid embeddings for {name}")

    # Save to disk
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "wb") as f:
        pickle.dump(database, f)

    print(f"\n💾 Database saved to {save_path}")
    print(f"👥 Registered persons: {list(database.keys())}")
    return database


# ── Run ─────────────────────────────────────────────────────
database = build_embeddings_database()


## Step 8 — Load Embeddings Database

In [ ]:
def load_database(path: str = "embeddings/face_embeddings.pkl") -> dict:
    if not Path(path).exists():
        print(f"❌ Database not found at {path}. Run Step 7 first.")
        return {}
    with open(path, "rb") as f:
        db = pickle.load(f)
    print(f"✅ Database loaded | Persons: {list(db.keys())}")
    return db


database = load_database()


## Step 9 — Recognition Engine

In [ ]:
def recognise_face(test_embedding: np.ndarray,
                   database: dict,
                   threshold: float = 0.45) -> tuple[str, float]:
    """
    Compare test embedding against all database entries.
    Returns (name, score). name='Unknown' if below threshold.
    Threshold tuning:
        0.45 → strict (fewer false positives)
        0.35 → relaxed (more matches but risk of false positives)
    """
    if not database:
        return "Unknown", 0.0

    scores = {name: cosine_score(test_embedding, emb)
              for name, emb in database.items()}
    best_name = max(scores, key=scores.get)
    best_score = scores[best_name]

    if best_score >= threshold:
        return best_name, best_score
    return "Unknown", best_score


def annotate_frame(frame: np.ndarray,
                   database: dict,
                   threshold: float = 0.45) -> np.ndarray:
    """Detect all faces in frame and annotate with name + score."""
    output = frame.copy()
    boxes = detect_faces_mtcnn(frame)

    for (x, y, w, h, conf) in boxes:
        x, y = max(0, x), max(0, y)
        face_crop = frame[y:y+h, x:x+w]

        if face_crop.size == 0:
            continue

        emb = get_arcface_embedding(face_crop)
        if emb is None:
            label, color = "No embedding", (128, 128, 128)
        else:
            name, score = recognise_face(emb, database, threshold)
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            label = f"{name} ({score:.2f})"

        # Draw bounding box
        cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)

        # Label background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(output, (x, y - th - 8), (x + tw + 4, y), color, -1)
        cv2.putText(output, label, (x + 2, y - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    return output


print("✅ Recognition engine ready")


## Step 10 — Test Recognition on a Static Image

In [ ]:
def test_on_image(image_path: str, database: dict, threshold: float = 0.45):
    """Load image, recognise faces, display and save annotated result."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Cannot read image: {image_path}")
        return

    result = annotate_frame(img, database, threshold)

    # Save output
    out_path = Path("output") / ("result_" + Path(image_path).name)
    cv2.imwrite(str(out_path), result)

    # Display in notebook
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    from IPython.display import display
    display(Image.fromarray(result_rgb))
    print(f"✅ Result saved to {out_path}")


# ── Place your test image in test_images/ and update the path ──
# test_on_image("test_images/test.jpg", database, threshold=0.45)

print("ℹ️  Uncomment the test_on_image() line above and provide a valid image path.")


## Step 11 — Real-Time Webcam Recognition

In [ ]:
def run_realtime_recognition(database: dict, threshold: float = 0.45):
    """
    Live webcam face recognition loop.
    Press Q or ESC to exit.
    """
    if not database:
        print("❌ Database is empty. Run Step 7 first.")
        return

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Webcam not available in this environment. Use the Streamlit app instead.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("🎥 Webcam started — press Q or ESC to quit.")
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Process every other frame to reduce CPU load
        if frame_count % 2 == 0:
            annotated = annotate_frame(frame, database, threshold)
        else:
            annotated = frame

        cv2.putText(annotated, "Press Q/ESC to quit", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.imshow("Face Recognition — Live", annotated)

        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):
            break

        frame_count += 1

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Webcam closed.")


# ── Uncomment to run ──
# run_realtime_recognition(database, threshold=0.45)

print("ℹ️  Uncomment run_realtime_recognition() above to start the live webcam feed.")


## Step 12 — Evaluate Recognition Accuracy

In [ ]:
def evaluate_accuracy(dataset_dir: str = "dataset",
                       database: dict = None,
                       threshold: float = 0.45,
                       test_split: float = 0.2):
    """
    Hold-out evaluation: 80 % train (already in database), 20 % test.
    Reports per-class and overall accuracy.
    """
    if database is None or not database:
        print("❌ Load/build database first.")
        return

    dataset_path = Path(dataset_dir)
    correct, total = 0, 0
    per_class = {}

    for person_dir in dataset_path.iterdir():
        if not person_dir.is_dir():
            continue
        name = person_dir.name
        images = list(person_dir.glob("*.jpg")) + \
                 list(person_dir.glob("*.jpeg")) + \
                 list(person_dir.glob("*.png"))

        # Use last 20 % as test set
        n_test = max(1, int(len(images) * test_split))
        test_imgs = images[-n_test:]

        class_correct = 0
        for img_path in test_imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            emb = get_arcface_embedding(img)
            if emb is None:
                continue
            pred_name, _ = recognise_face(emb, database, threshold)
            if pred_name == name:
                class_correct += 1
            total += 1
        correct += class_correct
        per_class[name] = (class_correct, len(test_imgs))

    print("\n📊 Evaluation Results")
    print("─" * 40)
    for name, (c, t) in per_class.items():
        pct = 100 * c / t if t else 0
        print(f"  {name:20s}: {c}/{t}  ({pct:.1f}%)")
    print("─" * 40)
    overall = 100 * correct / total if total else 0
    print(f"  {'Overall':20s}: {correct}/{total}  ({overall:.1f}%)")
    return overall


# ── Run evaluation ──
# evaluate_accuracy(database=database, threshold=0.45)

print("ℹ️  Uncomment evaluate_accuracy() above after building the database.")


## Step 13 — Enroll a New Person (Without Retraining)

In [ ]:
def enroll_person(name: str,
                  image_paths: list,
                  database: dict,
                  db_path: str = "embeddings/face_embeddings.pkl",
                  augment: bool = True) -> dict:
    """Add a new person to the database without reprocessing existing persons."""
    embeddings_list = []

    for path in tqdm(image_paths, desc=f"Enrolling {name}"):
        img = cv2.imread(str(path))
        if img is None:
            continue
        emb = get_arcface_embedding(img)
        if emb is not None:
            embeddings_list.append(emb)
        if augment:
            emb_f = get_arcface_embedding(cv2.flip(img, 1))
            if emb_f is not None:
                embeddings_list.append(emb_f)

    if not embeddings_list:
        print(f"❌ No valid face found in provided images for {name}.")
        return database

    database[name] = normalize([np.mean(embeddings_list, axis=0)])[0]

    with open(db_path, "wb") as f:
        pickle.dump(database, f)

    print(f"✅ {name} enrolled ({len(embeddings_list)} embeddings). Database updated.")
    return database


# Usage example:
# new_images = list(Path("dataset/NewPerson").glob("*.jpg"))
# database = enroll_person("NewPerson", new_images, database)

print("ℹ️  Use enroll_person() to add new identities without rebuilding the full database.")


## ✅ Summary — What Was Built

| Component | Technology | Purpose |
|---|---|---|
| Face Detection | **MTCNN** | Detect face bounding boxes with landmarks |
| Face Recognition | **ArcFace (buffalo_l)** | Generate 512-d face embeddings |
| Preprocessing | **CLAHE + resize** | Lighting robustness |
| Augmentation | **Horizontal flip** | Double training data |
| Matching | **Cosine Similarity** | Fast identity comparison |
| Storage | **Pickle (.pkl)** | Lightweight embedding database |
| UI | **Streamlit** | Web-based demo |

**Accuracy improvements over baseline:**
- `buffalo_l` model (highest insightface accuracy) instead of default
- CLAHE histogram equalisation for different lighting conditions  
- Mean-embedding aggregation per person for robustness  
- L2 normalised embeddings for stable cosine comparisons  
- Augmentation (horizontal flip) for better coverage  
- Confidence filtering (≥ 0.95) in MTCNN for clean training data  

---
> Run `streamlit run app.py` to launch the web UI.
